# US 4.5 -- Multi-Site Evaluation

After training both the baseline and DANN models, we evaluate them on
**all domains** (including sites never seen during training) to quantify
the improvement from domain adaptation.

## What you will learn

1. How to load a trained checkpoint
2. How to evaluate on multiple domains using `evaluate_all_domains`
3. How to build a comparison table: baseline vs. DANN

## CLI equivalent

```bash
udm-epic4 evaluate --checkpoint outputs/epic4_dann/best.pth \
                    --config configs/epic4_evaluate.yaml
```

In [ ]:
import torch
import yaml
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

from udm_epic4.models.dann import DANNModel
from udm_epic4.data.multi_domain_dataset import DomainDataset
from udm_epic4.evaluation.metrics import (
    compute_f1,
    compute_iou,
    compute_dice,
    evaluate_all_domains,
)

---
## 1. Loading a Trained Checkpoint

Checkpoints are saved as dictionaries with these keys:

| Key | Description |
|-----|------------|
| `model_state_dict` | Model weights (for DANN checkpoints) |
| `encoder_state_dict` | Encoder weights (for baseline checkpoints) |
| `decoder_state_dict` | Decoder weights (for baseline checkpoints) |
| `optimizer_state_dict` | Optimizer state |
| `epoch` | Epoch at which the checkpoint was saved |
| `val_f1` | Validation F1 at checkpoint time |

In [ ]:
def load_dann_model(checkpoint_path, device='cpu'):
    """Load a DANNModel from a checkpoint file."""
    model = DANNModel(
        backbone='convnext_tiny',
        pretrained=False,              # weights come from checkpoint
        decoder_channels=[256, 128, 64, 32],
        domain_head_hidden=256,
    )
    state = torch.load(str(checkpoint_path), map_location=device, weights_only=False)

    # Handle both DANN and baseline checkpoint formats
    if 'model_state_dict' in state:
        model.load_state_dict(state['model_state_dict'])
    else:
        model.load_state_dict(state)

    model.to(device).eval()
    print(f"Loaded checkpoint from epoch {state.get('epoch', '?')}, "
          f"val_f1={state.get('val_f1', '?')}")
    return model


# Usage:
# model = load_dann_model('outputs/epic4_dann/best.pth', device='cuda')
print("(Checkpoint loading ready -- provide a real checkpoint path to use.)")

---
## 2. Evaluating on All Domains

The evaluation config (`configs/epic4_evaluate.yaml`) lists all domains to test:

```yaml
evaluation:
  - name: warstein
    images: data/warstein/val/images
    masks: data/warstein/val/masks
  - name: malaysia
    images: data/malaysia_eval/images
    masks: data/malaysia_eval/masks
  - name: regensburg
    images: data/regensburg/images
    masks: data/regensburg/masks
  - name: china
    images: data/china/images
    masks: data/china/masks
```

The `evaluate_all_domains` function runs inference on each dataset and
returns a DataFrame with F1, IoU, and Dice per domain.

In [ ]:
# Full evaluation workflow (with real data):
#
# with open('configs/epic4_evaluate.yaml') as f:
#     cfg = yaml.safe_load(f)
#
# eval_datasets = []
# for i, ev in enumerate(cfg['evaluation']):
#     ds = DomainDataset(ev['images'], ev.get('masks'), domain_label=i,
#                        image_size=(512, 512))
#     eval_datasets.append(ds)
#
# results_df = evaluate_all_domains(model, eval_datasets, device='cuda')
# print(results_df)
#
# Save to CSV:
# results_df.to_csv('outputs/epic4_evaluation/results.csv', index=False)

print("Evaluation workflow ready. Provide real data paths to run.")

---
## 3. Baseline vs. DANN Comparison

The key deliverable is a side-by-side comparison showing that DANN improves
target-domain performance while maintaining source-domain quality.

In [ ]:
# Simulated comparison results (replace with actual evaluation output)
comparison_data = {
    'Domain': ['warstein (source)', 'malaysia (target)', 'regensburg (target)', 'china (target)'],
    'Baseline F1': [0.87, 0.58, 0.52, 0.55],
    'DANN F1':     [0.85, 0.74, 0.69, 0.72],
    'Baseline IoU': [0.77, 0.41, 0.35, 0.38],
    'DANN IoU':     [0.74, 0.59, 0.53, 0.56],
}
df = pd.DataFrame(comparison_data)

# Compute improvement
df['F1 Delta'] = df['DANN F1'] - df['Baseline F1']
df['IoU Delta'] = df['DANN IoU'] - df['Baseline IoU']

print("=" * 70)
print("Baseline vs. DANN -- Segmentation Performance")
print("=" * 70)
print(df.to_string(index=False))
print()

# Averages
target_mask = df.index > 0
print(f"Mean target F1 improvement: {df.loc[target_mask, 'F1 Delta'].mean():+.2f}")
print(f"Mean target IoU improvement: {df.loc[target_mask, 'IoU Delta'].mean():+.2f}")

In [ ]:
# Grouped bar chart
domains = df['Domain'].values
x = np.arange(len(domains))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))

bars1 = ax.bar(x - width/2, df['Baseline F1'], width, label='Baseline (source-only)',
               color='#90CAF9', edgecolor='white')
bars2 = ax.bar(x + width/2, df['DANN F1'], width, label='DANN',
               color='#2196F3', edgecolor='white')

# Value labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)

ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('Baseline vs. DANN -- F1 per Domain', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(domains, fontsize=10)
ax.set_ylim(0, 1.0)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
fig.tight_layout()
plt.show()

print("DANN trades a small source-domain decrease for large target-domain gains.")

---
## 4. Available Metrics

The `udm_epic4.evaluation.metrics` module provides three pixel-level metrics:

| Metric | Function | Description |
|--------|---------|------------|
| **F1** | `compute_f1(pred, target)` | Harmonic mean of precision and recall |
| **IoU** | `compute_iou(pred, target)` | Intersection over Union (Jaccard) |
| **Dice** | `compute_dice(pred, target)` | Identical to F1 for binary masks |

All three accept raw logits or probabilities and apply sigmoid + thresholding
internally.

In [ ]:
# Quick metrics demo on synthetic data
pred = torch.randn(1, 1, 64, 64)  # raw logits
target = (torch.rand(1, 1, 64, 64) > 0.7).float()  # binary mask

f1  = compute_f1(pred, target)
iou = compute_iou(pred, target)
dice = compute_dice(pred, target)

print(f"F1:   {f1:.4f}")
print(f"IoU:  {iou:.4f}")
print(f"Dice: {dice:.4f}")
print(f"F1 == Dice: {abs(f1 - dice) < 1e-6}")  # True for binary masks

---
## Summary

- Load checkpoints with `torch.load` and restore model weights.
- `evaluate_all_domains()` returns a DataFrame with per-domain F1/IoU/Dice.
- Compare baseline vs. DANN to quantify domain adaptation benefit.
- Expected: small source decrease, large target improvement.

**Next:** `epic4_06_failure_analysis.ipynb` -- deep-dive into failure modes.